# Domain Adaptation and Interpretability in Character-Level Language Models

This notebook documents my experiments training NanoGPT-style language models on multiple text domains, measuring cross-domain transfer, and inspecting token-level behavior through analysis and visualization.


## Project Overview

This project explores how small character-level transformer models specialize across different text domains and how that specialization affects transfer. I trained NanoGPT-style models on Shakespeare, Wikipedia, and math text, then compared in-domain quality, cross-domain behavior, few-shot adaptation, and token-level attribution signals.


## Tech Stack

- Python
- PyTorch
- NumPy
- Matplotlib
- custom NanoGPT-style model code


## Training Domain-Specific Models

* Use the same small NanoGPT config (e.g., `n_layer=2`, `n_head=2`, `n_embd=128`, `block_size=64`) for all runs.
* Train on CPU or free Colab GPU — each model should converge in minutes given the small corpora and reduced parameters.
* Monitor:

* Training loss curves.
* Qualitative performance of generated samples.

This section mostly requires you to run scripts that are already written. At each step, pay close attention to the comments, as you may be asked about them in the future.


In [ ]:
import os, io, zipfile, requests
from pathlib import Path

# Choose the corpus
corpus_name = "shakespeare"  # Change to "shakespeare", "wikipedia", or "math"

# Reliable sources:
# - shakespeare: Karpathy's tiny Shakespeare (plain text)
# - wikipedia: https://www.kaggle.com/datasets/ffatty/plain-text-wikipedia-simpleenglish
# - math: https://archive.org/stream/CalculusMadeEasy/Calculus_Made_Easy_Thompson_djvu.txt
 


# Download and show sample
 
text = Path(f"data/{corpus_name}.txt").read_text(encoding="utf-8", errors="ignore")
print(f"Corpus length: {len(text)} characters")
print("Sample:")
print(text[:300])


In [ ]:
# -----------------------------------------
# Prepare dataset for character-level modeling
# -----------------------------------------

import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from torch.utils.data import DataLoader

class CharDataset(Dataset):
    def __init__(self, text, block_size, stoi=None, itos=None):
        """
        text: The raw text string we want to train on.
        block_size: The length of each training sequence (number of characters).
        stoi, itos: Optional vocab mappings. If provided, reuse them.
        """

        # 1. Build or reuse the vocabulary
        if stoi is None or itos is None:
            # Build from scratch
            self.chars = sorted(list(set(text)))
            self.vocab_size = len(self.chars)
            self.stoi = {ch: i for i, ch in enumerate(self.chars)}
            self.itos = {i: ch for i, ch in enumerate(self.chars)}
        else:
            # Reuse given vocab
            self.stoi = stoi
            self.itos = itos
            self.vocab_size = len(self.stoi)

        # 2. Store sequence length
        self.block_size = block_size

        # 3. Encode dataset into indices
        #    Use .get(ch, 0) so unknown characters map to 0
        self.data = torch.tensor([self.stoi.get(c, 0) for c in text], dtype=torch.long)

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        chunk = self.data[idx : idx + self.block_size + 1]
        x = chunk[:-1]
        y = chunk[1:]
        return x, y


block_size = 64
text = Path(f"data/{corpus_name}.txt").read_text(encoding="utf-8", errors="ignore")
dataset = CharDataset(text, block_size=block_size, stoi=None, itos=None)
 
 


## Analysis: Character Frequency Histogram

Visualize the differences in vocabulary and character usage between corpora (`shakespeare`, `wikipedia`, `math`). Do this by creating a histogram of each character's frequency of occurance, for each corpora.

Unicode safety

* Some characters may not render properly in plots or Jupyter.
* Replace problematic characters with their Unicode code point.
* Example helper function:

```python
def safe_label(c):
try:
c.encode("ascii")  # check if ASCII-printable
return c
except UnicodeEncodeError:
return f"U+{ord(c):04X}"
```


* Generate one histogram for each corpus (`shakespeare`, `wikipedia`, `math`).
* For each corpus, write 1–2 sentences describing:

* Which characters are most common.
* Any unusual symbols or formatting.
* How the distributation of the corpus represents a distribution of a more general language model, e.g. one learned by ChatGPT.


In [ ]:
import collections, matplotlib.pyplot as plt
from pathlib import Path

def safe_label(c):
    try:
        c.encode("ascii")  # check if ASCII-printable
        return c
    except UnicodeEncodeError:
        return f"U+{ord(c):04X}"

corpora = {
    "shakespeare": "data/shakespeare.txt",
    "wikipedia": "data/wikipedia.txt",
    "math": "data/math.txt",
}

# Plotting histogram for each corpora 
for name, path in corpora.items():
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()
    
    chars = []
    for ch in text:
        if not ch.isspace():
            chars.append(ch)                
    counter = collections.Counter(chars)
    common = counter.most_common(50)
    
    labels = [safe_label(c) for c, _ in common]
    values = [v for _, v in common]

    plots_dir = Path("plots")
    plots_dir.mkdir(parents=True, exist_ok=True)
    
    plt.figure(figsize=(10, 4))
    plt.bar(range(len(values)), values)
    plt.xticks(range(len(labels)), labels, rotation=75, ha="right")
    plt.title(f"Top 50 Character Frequencies in {name.capitalize()} Corpus")
    plt.xlabel("Character")
    plt.ylabel("Frequency")

    out_png = plots_dir / f"char_freq_top50_{name}.png"
    plt.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.show()

## Shakespeare Corpus:
- Most common characters: e, t, o, a, h, s, r, n, i, l.
- Unusual symbols:
- Heavy punctuation: :, ., ’, ?, ! — reflecting dramatic dialogue.
- Uppercase letters: A, T, E, W, H, M, B — from character names and stage directions.
- Distribution meaning: Shakespeare’s text is more literary, dialogue-driven, and dramatic. A model trained on this alone would learn archaic phrasing, lots of dialogue markers, and a higher-than-normal punctuation usage compared to modern text.

## Wikipedia Corpus:
- Most common characters: Same English backbone: e, a, t, o, n, i, s, r, h, l.
- Unusual symbols:
- Numbers (0–9) are frequent — because of dates, references, and factual details.
- Quotation marks (") are unusually high due to citations.
- Parentheses and punctuation appear consistently ((, ), ,, .).
- Distribution meaning: The style is factual, reference-heavy, and neutral. A model trained primarily on this would learn formal, citation-oriented prose, suitable for encyclopedic explanations.

## Math Corpus:
- The most common characters are e, t, a and i.
- The frequent use of mathematical signs like +, = and ^ stands out the most when compared to other corporas.
- The distribution suggests that this data is best suited for domain-specific tasks, in this case, mathematics.


In [ ]:
from nanogpt_model import GPT  # Import the GPT model class from the NanoGPT repo


# -----------------------------------------
# Define a configuration object for the GPT model
# -----------------------------------------
class GPTConfig:
    def __init__(self, vocab_size, block_size,
                 n_layer=2, n_head=2, n_embd=128, dropout=0.0, bias = True):
        """
        This class is a container for all the key hyperparameters
        that define the size and shape of the GPT model.
        We pass this config into the GPT constructor so the model
        can be built with these exact settings.

        ---------------------------
        Parameter meanings:
        ---------------------------

        vocab_size:
            - Number of unique tokens in our dataset.
            - In a character-level model, this is the number of distinct characters.
            - Maps directly to the size of the token embedding matrix:
                token_embedding_table.shape == (vocab_size, n_embd)

        block_size:
            - Maximum context length (sequence length) the model sees at once.
            - Sets the width of the positional embedding table:
                position_embedding_table.shape == (block_size, n_embd)
            - Also defines the mask size in self-attention so the model only attends
              to the last `block_size` tokens.

        n_layer:
            - Number of Transformer blocks stacked in the model.
            - Each block = (Multi-Head Self-Attention + Feedforward MLP + LayerNorm).
            - More layers allow the model to capture more complex dependencies.

        n_head:
            - Number of attention heads per multi-head attention layer.
            - Each head learns to focus on different positions/tokens in the sequence.
            - Heads are concatenated then projected back into `n_embd` dimensions.

        n_embd:
            - Dimensionality of token embeddings and all hidden states.
            - Controls the "width" of the model.
            - Affects:
                * Token embedding size
                * Positional embedding size
                * Per-head dimension in attention: head_dim = n_embd / n_head
                * Hidden size of the feedforward layers in each block.

        dropout:
            - Dropout probability applied in various places (attention, MLP) during training.
            - Helps prevent overfitting by randomly zeroing activations.
 
        """

        # Store all parameters for use by the GPT class
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.n_layer = n_layer
        self.n_head = n_head
        self.n_embd = n_embd
        self.dropout = dropout 
        self.bias = bias  


config = GPTConfig(vocab_size=dataset.vocab_size, block_size=block_size)
model = GPT(config)
print(model)

 

## Analysis: Model Architecture Diagram

Create a clear, labeled diagram of the NanoGPT model you trained, showing the flow of data from input to output, including the shapes/dimensions at each stage. Do this *by hand*. You may use a tablet, but it cannot look code-generated.


1. Overall Layout

* Show the full pipeline:
Input Tokens → Token Embedding → Positional Embedding → Transformer Blocks → LayerNorm → Output Head → Softmax.

2. Transformer Blocks

* Each block should include:

* LayerNorm 1 → Multi-Head Causal Self-Attention → Residual Connection
* LayerNorm 2 → Feedforward MLP → Residual Connection
* Clearly label these sub-components and indicate that this block repeats `n_layer` times.

3. Dimensions

* Label the shape of the tensor at each stage (batch size = `B`, sequence length = `T`, embedding size = `n_embd`):

* Input: `(B, T)` (integer token IDs)
* Token embeddings: `(B, T, n_embd)`
* After adding positional embeddings: `(B, T, n_embd)`
* Inside attention: queries/keys/values → `(B, n_head, T, head_dim)`
* MLP layers: `(B, T, 4*n_embd)` then back to `(B, T, n_embd)`
* Output logits: `(B, T, vocab_size)`

4. Connections

* Draw arrows between each component to show the data flow.
* Mark residual connections that add the block’s input to its output.

5. Label Hyperparameters

* Include:

* `n_layer` = number of transformer blocks
* `n_head` = number of attention heads per block
* `n_embd` = embedding dimension
* `block_size` = maximum context length
* `vocab_size` = size of the token vocabulary

6. Final Output

* Show that the model outputs logits for each token position, then a softmax over the vocabulary to get probabilities.


![NanoGPT](../assets/model_architecture_reference.jpeg)

## Training NanoGPT

The following code actually trains NanoGPT on your selected dataset.
Your job is to:

1. Play around with the hyperparameters (e.g. learning rate, batch size, context length, number of layers).
2. Run the training loop long enough to get meaningful results.
3. Save a trained checkpoint for each of the three corpora (`shakespeare`, `wikipedia`, `math`).

After training:

* You should have a folder of checkpoints (e.g. `checkpoints/shakespeare/final.pt`).
* Each corpus will give you a slightly different “voice” when you generate text.


In [ ]:
import os
import torch
from tqdm import tqdm
print(corpus_name)
# Directory for this corpus
ckpt_dir = os.path.join("checkpoints", corpus_name)
os.makedirs(ckpt_dir, exist_ok=True)
final_ckpt_path = os.path.join(ckpt_dir, "final.pt")

if os.path.exists(final_ckpt_path):
    print(f"Final checkpoint for '{corpus_name}' found at {final_ckpt_path}.")
    ckpt = torch.load(final_ckpt_path, map_location="cpu")
    model.load_state_dict(ckpt["model_state"])
    losses = ckpt.get("losses", [])
    print(f"Loaded model with {len(losses)} stored loss values.")
else:
    print(f"No checkpoint found for '{corpus_name}', starting training...")
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    loader = DataLoader(dataset, batch_size=32, shuffle=True)
    max_iters = 5000

    losses = []   # will now store only full-batch eval losses
    model.train()
    for it in range(max_iters):
        xb, yb = next(iter(loader))

        logits, loss = model(xb, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if it % 100 == 0:
            # Evaluate on full dataset (avg loss across all batches)
            model.eval()
            eval_losses = []
            
            max_batches = len(loader) * 0.01
            for i, (xb_eval, yb_eval) in enumerate(loader):
                if i >= max_batches:  break
                logits_eval, loss_eval = model(xb_eval, yb_eval)
                eval_losses.append(loss_eval.item())
            avg_loss = torch.mean(torch.tensor(eval_losses))
            losses.append(avg_loss)
            print(f"Iter {it}/{max_iters}, Avg Loss over dataset: {avg_loss:.4f}")

            # Save periodic checkpoint
            iter_ckpt_path = os.path.join(ckpt_dir, f"iter_{it}.pt")
            torch.save({
                "model_state": model.state_dict(),
                "config": config.__dict__,
                "itos": dataset.itos,
                "stoi": dataset.stoi,
                "losses": losses
            }, iter_ckpt_path)
            print(f"Saved checkpoint: {iter_ckpt_path}")
            model.train()  # back to training mode

    # Save final checkpoint
    torch.save({
        "model_state": model.state_dict(),
        "config": config.__dict__,
        "itos": dataset.itos,
        "stoi": dataset.stoi,
        "losses": losses
    }, final_ckpt_path)
    print(f"Saved final model to {final_ckpt_path}")


## Analysis: Track Model Output Over Training

I compare checkpoint snapshots across training to see when each model begins producing coherent domain-specific text and how generation quality evolves over time.


In [ ]:
import os

# This code loads a specific training checkpoint of NanoGPT
# (saved during training) and uses it to generate text.


def generate_small_text(corpus, start_text, iteration_to_load, max_new_tokens):
    # Path to the specific checkpoint
    
    if iteration_to_load == "final":
        ckpt_path = os.path.join("checkpoints", corpus, "final.pt")
    else:
        ckpt_path = os.path.join("checkpoints", corpus, f"iter_{iteration_to_load}.pt")
    
    if not os.path.exists(ckpt_path):
        print(f"Checkpoint {ckpt_path} not found. Make sure to train first.")
    else:
        # Load checkpoint from disk
        ckpt = torch.load(ckpt_path, map_location="cpu")
    
        # Rebuild the GPT model from saved config
        loaded_config = type("GPTConfig", (), ckpt["config"])
        gen_model = GPT(loaded_config)
        gen_model.load_state_dict(ckpt["model_state"])
        gen_model.eval()
    
        # Tokenization: convert characters to IDs
        stoi = ckpt["stoi"]
        itos = ckpt["itos"]
        idx = torch.tensor([stoi.get(c, 0) for c in start_text], dtype=torch.long).unsqueeze(0)
    
        # Autoregressive generation loop
        with torch.no_grad():
            for _ in range(max_new_tokens):
                # Feed only the last block_size tokens
                idx_cond = idx[:, -loaded_config.block_size:]
                logits, _ = gen_model(idx_cond)
    
                # Convert logits to probabilities
                probs = torch.softmax(logits[:, -1, :], dim=-1)
    
                # Sample next token from probability distribution
                idx_next = torch.multinomial(probs, num_samples=1)
    
                # Append new token to the sequence
                idx = torch.cat((idx, idx_next), dim=1)
    
        # Decode IDs back into text
        generated_text = ''.join([itos[i.item()] for i in idx[0]])
        return generated_text

In [ ]:
corpus_list = ['shakespeare', 'wikipedia', 'math']
prompt = 'Once upon a time'
iteration_list = [0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900, 2000, 2100, 2200, 2300, 2400, 2500, 2600, 2700, 2800, 2900, 3000, 3100, 3200, 3300, 3400, 3500, 3600, 3700, 3800, 3900, 4000, 4100, 4200, 4300, 4400, 4500, 4600, 4700, 4800, 4900, 'final']
max_tokens = 100

# Saving generated output in output folder under each corpora
for corpus in corpus_list:
    output_file = os.path.join("output", corpus+".txt")
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, "a", encoding="utf-8") as f:
        for iteration in iteration_list:
            try:
                line = generate_small_text(corpus, prompt, iteration, max_tokens)
                f.write(f"{corpus}\t{iteration}\t{line}\n")
            except FileNotFoundError:
                f.write(f"{corpus}\t{iteration}\t[MISSING_CHECKPOINT]\n")

## Shakespeare Corpus

| ITERATION | OUTPUT                                                                                                                                      |
|-----------|---------------------------------------------------------------------------------------------------------------------------------------------|
| 100       | Once upon a time ws: <br> IYof er padiivisy hig sdr meaveakdes. <br> Mice, <br> QU! ayord f, syo pour tou; d lyoil ou ate th'dou ir          |
| 500       | Once upon a time tas the Nor <br> you all gratakent tith came will I prroth? <br><br> Cah, lOow! <br><br> TKENTIUCH: <br> Aull is make friover |
| 1000      | Once upon a time: <br> Helquees thraction! 'I'll that a pusuble <br> goody lance an sheir orme <br> To but Marry, worst nor his dea          |
| 2000      | Once upon a time, passip! <br> Neith, should be strenge, no! Comether, I well. I my faren <br> Thou gracentling execution. <br><br> RO       |
| 3000      | Once upon a time apear-laff: <br><br> POLIXENENIUS: <br> My head more, my see! <br><br> KING RICHARD III:: <br> Alas it goars, and me to king. |
| 4000      | Once upon a time oather befits and alty, <br> And meet enter and for wrong. I had, from the good tender, <br> That I have heat               |
| FINAL     | Once upon a times and toor a false and grace. <br><br> Good liege: <br> A men, it is some am unto the king! <br> In London the firms c       |

- Coherence: By ~2000 iterations, dialogue-like phrasing appears. By 3000, explicit names and stage directions (POLIXENENIUS, KING RICHARD III) emerge.
- Style: Strongly dramatic, archaic, with heavy punctuation and stage cues.
- Overconfidence: At 4000 and FINAL, outputs recycle formulaic Shakespeare tropes (“Good liege”, “Alas”, “In London”), showing memorization of the corpus style.

## Wikipedia Corpus

| ITERATION | OUTPUT                                                                                                                                     |
|-----------|--------------------------------------------------------------------------------------------------------------------------------------------|
| 100       | Once upon a timeceg tiis phaltamesenrorolos T Dacy hen war wanPes se st om ome n 'vetNitoerof pelocumice CFses mppkt                       |
| 500       | Once upon a timenos.–, Nthep's man campes alalay adion isupers corts ra axock ethóʻe boeks or enlless baldats. Carme                      |
| 1000      | Once upon a time of rine is owll spystivel is fult lrued found Inlian be perimal say. Eus seaese wis hudy ofle thele                      |
| 2000      | Once upon a time, invanG extebels) and fror for can did trift proforts. Aboth perfoectly. It is other por very than                        |
| 3000      | Once upon a time is galaxies, the can needul ban size dayhou's measure by procent year Before can galaxies implate m                       |
| 4000      | Once upon a time, in alphabetic enjocine conside what parks buynny in Gaving (energy of the temperature, the felity.                       |
| FINAL     | Once upon a time astrefards, who makes money are a rical of and. The nations. London, energed the United King langua                       |

- Coherence: By ~2000 iterations, grammar is forming (“perfectly… It is other…”). By 3000, it produces factual-sounding content (galaxies, measurements).
- Style: Encyclopedic, scientific, fact-heavy. Later checkpoints resemble Wikipedia’s neutral, explanatory prose.
- Overconfidence: By FINAL, it starts repeating stiff patterns (“The nations. London… United King…”), showing memorization of article-style openings.

## Math Corpus

| ITERATION | OUTPUT                                                                                                                                     |
|-----------|--------------------------------------------------------------------------------------------------------------------------------------------|
| 100       | Once upon a timererffowtithedere w ch ■ ong y amaler tthererasin ouher thet cind? veaves eJ ble be sof corelof,T                           |
| 500       | Once upon a time forswn ppum boinste it is withulton. The tocke re th is incath betres andd tthe sec bre yinsh                             |
| 1000      | Once upon a time at called godget dby; one there, elogatt y and them the are pid g, and Then a? or, hencesss wher                          |
| 2000      | Once upon a time-mennsion mather goots of the Orgiding Qy : the averify by the boesed and on infiner we ar; mand t                         |
| 3000      | Once upon a time constant integrate partial france - log e this from tors at all lenot a whold or x. But beginn our                        |
| 4000      | Once upon a time size so zero, so work such as to called know from the inted not einder about this cases it does :                         |
| FINAL     | Once upon a time. Also of this chapter we obtained " and on't 5 hthe same efficiency, that is the various more of                          |

- Coherence: At ~2000 iterations, sentences reference mathematical reasoning (“dimension… verify… infinite”). At 3000, specific math terms appear (“integrate partial”, “log e”, “x”).
- Style: Technical, equation-like, with fragments resembling textbook theorems.
- Overconfidence: Later checkpoints sound formulaic, like boilerplate math proofs (“Also of this chapter we obtained…”), rather than flexible explanations.


## Analysis: Variability of the Final Model

I sample repeatedly from the final checkpoint for each corpus to measure output diversity, repetition, and stylistic consistency.


In [ ]:
corpus_list = ['shakespeare', 'wikipedia', 'math']
prompt = 'Once upon a time'
max_tokens = 100

# Saving generated output in final_output folder under each corpora
for corpus in corpus_list:
    output_file = os.path.join("final_output", corpus+".txt")
    os.makedirs(os.path.dirname(output_file), exist_ok=True)
    with open(output_file, "a", encoding="utf-8") as f:
        for _ in range(5):
            try:
                line = generate_small_text(corpus, prompt, 'final', max_tokens)
                f.write(f"{corpus}\t{iteration}\t{line}\n")
            except FileNotFoundError:
                f.write(f"{corpus}\t{iteration}\t[MISSING_CHECKPOINT]\n")

## Shakespeare Corpus

| Iteration | Output                                                                                                                              |
|-----------|-------------------------------------------------------------------------------------------------------------------------------------|
| Final     | Once upon a time my mind loss, Green their great tears set pretest contents, Why this vey show which tortuned a! And                 |
| Final     | Once upon a time oat. GLOUCESTER: Good, hence, I speak, fer'd her you first. See his gream behollow the daughter? S                  |
| Final     | Once upon a time, as I died Richard's shame you Eve, my cord schant to our want. VOLERNCENTIO: No Butchere as that                   |
| Final     | Once upon a time and our prophecased he come, As I, Mistrates Was to joy'st: I beseech it so. HORTENSIO: So assist                   |
| Final     | Once upon a time and my times hearts Froth, good call informity; orransaling nowes anot. KING RICHARD III: Farewell                  |

- Variance: The generations vary in characters, scenes, and lines (Gloucester, Richard, Hortensio). Different dramatic contexts appear.
- Style capture: Very strong. It nails Shakespeare’s dialogue conventions: named characters, archaic phrasing, emotional tone. Even if the words are jumbled, the cadence is unmistakable.
- Satisfaction: Yes — these outputs feel Shakespearean. The model learned both form (dialogue, stage names) and tone (dramatic, archaic). Semantic coherence is weak, but stylistic mimicry is clear.

## Wikipedia Corpus

| Iteration | Output                                                                                                                              |
|-----------|-------------------------------------------------------------------------------------------------------------------------------------|
| Final     | Once upon a time: Did no did not so it over at more cup Diviction or Schip itance, Egypt War Sea, which division wit                 |
| Final     | Once upon a time indely apple including they are called its (in good. Mercuryider momes than relybolations and other                 |
| Final     | Once upon a time people in future live in its. They a lot area- Cuban very colors of atbout 2st people are still on                  |
| Final     | Once upon a time doil, The United Sibnations, long, Birtomic African Yara (Jeaph treatly communic givining and mer                   |
| Final     | Once upon a time. Nolume may a biorightly a skmal sun Monque Dreplare Bern Russe for wanetable. The tensitures in C                  |

- Variance: The outputs reference different topics — Egypt, Mercury, Cuba, “United Sibnations”, even physics-like terms (“tensitures”). The content wanders but sticks to factual/explanatory domains.
- Style capture: Yes. It produces encyclopedia-like entries: lists of entities, geographic references, scientific-sounding terminology. The neutral tone and referential structure mirror Wikipedia’s style.
- Satisfaction: Reasonably good — while grammatically broken, the model consistently generates Wikipedia-like definitions/explanations. It shows the model internalized the format of articles (lists of facts, countries, science words).

## Math Corpus

| Iteration | Output                                                                                                                              |
|-----------|-------------------------------------------------------------------------------------------------------------------------------------|
| Final     | Once upon a time y = - (if centre case when the point of the total of the curve, from the ordinary curve; named wi                   |
| Final     | Once upon a time take the contact. Athe rist) of which one differentiating (0g+Ax-\ 2 )\ the dee-eks" fractions wh                   |
| Final     | Once upon a time at nevery the way instead of the time-constant to in Fig. 210. ~jWi^s «| id = 10; and, or every                     |
| Final     | Once upon a time is expressed by introded givided -"-7L-I - -j- - - + - r- -— . . On this : J takes then the radi                    |
| Final     | Once upon a time ttio simpleired y, work we see that this is to work heighth vious An example is x = ^x S +Sxdx =                    |

- Variance: The outputs vary — some look like textbook prose (“time-constant… Fig. 210”), others look like symbolic equations (“y = - …”, “x = ^x S + Sxdx”).
- Style capture: Yes, the model clearly picked up mathematical textbook style: equations embedded in English, references to figures, technical phrases.
- Satisfaction: For a small/early-trained model, this is impressive — it captures the flavor of math writing, even though the equations are nonsensical. It “learned” the surface form of math texts but not their correctness.


## Analysis: Training Loss Across Checkpoints

I plot one training curve per corpus and compare how quickly each model becomes useful in practice. The main goal is to connect quantitative loss values with qualitative generation quality rather than treating lower loss as sufficient on its own.

This analysis helps answer three project questions:

- when each domain begins producing coherent text
- whether different corpora require different convergence thresholds
- how loss dynamics relate to repetition, stability, and stylistic quality


In [ ]:
import os, torch, matplotlib.pyplot as plt
from pathlib import Path

iteration_list = [0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 1100, 1200, 1300, 1400, 1500, 1600, 1700, 1800, 1900, 2000, 2100, 2200, 2300, 2400, 2500, 2600, 2700, 2800, 2900, 3000, 3100, 3200, 3300, 3400, 3500, 3600, 3700, 3800, 3900, 4000, 4100, 4200, 4300, 4400, 4500, 4600, 4700, 4800, 4900, 'final']
checkpoint_dirs = {
    "shakespeare": "checkpoints/shakespeare",
    "wikipedia": "checkpoints/wikipedia",
    "math": "checkpoints/math",
}

# Taking the checkpoint
def get_ckpt_path(d, iter):
    if iter != "final":
        p = os.path.join(d, f"iter_{iter}.pt")
        return p if os.path.exists(p) else None
    p = os.path.join(d, "final.pt")
    return p if os.path.exists(p) else None

# Taking loss from checkpoint
def get_loss(ckpt):
    x = ckpt["losses"][-1]
    return float(x.item()) if hasattr(x, "item") else float(x)

series = {}  # To save the values of loss for different corporas

# Plotting Line graph for each corpora
for corpus, d in checkpoint_dirs.items():
    xs, ys = [], []
    for step in iteration_list:
        p = get_ckpt_path(d, step)
        if not p:
            print("Missing checkpoint. Please run the checkpoint code block first!")
            break
        try:
            ckpt = torch.load(p, map_location="cpu")
        except Exception as e:
            print("Missing checkpoint. Please run the checkpoint code block first!")
            break

        loss = get_loss(ckpt)
        if isinstance(step, int): 
            x = step 
        else: 
            x = (xs[-1] + 100) if xs else 0 
        xs.append(x); ys.append(loss); last_x = x

    series[corpus] = (xs, ys)

    plt.figure()
    if xs:
        plt.plot(xs, ys, marker="o")
    plt.title(f"{corpus.capitalize()} loss")
    plt.xlabel("Iteration")
    plt.ylabel("Loss")
    out_png = plots_dir / f"{corpus}_loss.png"
    plt.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.show()

# Plotting Line graph for comparision between corporas
plt.figure()
for corpus, (xs, ys) in series.items():
    if xs:
        plt.plot(xs, ys, label=corpus)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.legend()
out_png_cmp = plots_dir / "loss_comparison.png"
plt.savefig(out_png_cmp, dpi=200, bbox_inches="tight")
plt.show()

## When Loss Values Correspond to Good Performance
- Initial phase (0–200 iterations): Loss drops steeply but outputs are usually still incoherent. The model learns token-level patterns (syntax, common words).
- Mid phase (~1.8–1.5 loss): Text starts resembling domain-appropriate structure. For example, sentences become grammatical, math expressions look structured, or Wikipedia-like exposition appears.
- Later phase (<1.5 loss): This is when qualitatively good, convincing outputs emerge—coherence across multiple sentences, domain-appropriate style, and reduced nonsense.
→ So, loss <1.5 is the rough threshold where models start producing human-like, domain-specific text.

## Corpus-by-Corpus Differences
- ### Math
- Starts high (~4.3) but converges fastest, dropping below 1.3 by 5000 iterations.
- Quality likely required very low losses (<1.4) because mathematical notation is rigid and unforgiving—small errors produce nonsensical equations.
- By the end, the lowest loss among all three corpora, suggesting the model mastered predictable formulaic patterns.

- ### Shakespeare
- Starts at ~3.8 and steadily improves, plateauing around 1.4.
- Convincing Shakespearean text likely appeared earlier (around 1.5–1.6), because the style is highly repetitive and formulaic (rhythmic structures, common phrases).
- Did not reach as low a final loss as Math, but "sounded good" sooner.

- ### Wikipedia
- Starts highest (~5.2) and converges more slowly, flattening at ~1.45.
- Wikipedia requires broader vocabulary and factual consistency, so outputs sounded good at slightly higher loss (~1.6–1.7) than Math.
- Even at similar loss values, Wikipedia text is harder to judge “good” because coherence depends on factual grounding, not just style.


## Cross-Domain Evaluation

* Evaluate each trained model on each other’s dataset:

* Zero-shot: Direct prompt without domain examples.
* Few-shot: Include 1–3 in-domain examples in the prompt.
* Compare:

* Loss in-domain vs. out-of-domain.
* Quality and relevance of generated continuations.


In [ ]:
import torch, math
from pathlib import Path
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

corpora = ["shakespeare", "wikipedia", "math"]
prompts = {
    "shakespeare": "What's done can't be undone",
    "wikipedia": "This article provides that",
    "math": "Let's take an example of variable x",
}

# Return final checkpoint file for corpora
def final_checkpoint(corpus):
    try:
        return Path("checkpoints") / corpus / "final.pt"
    except:
        raise FileNotFoundError(f"final.pt not found for {corpus}")

# load model from final checkpoint of a corpus
def load_model(corpus):
    ckpt_path = final_checkpoint(corpus)
    ckpt = torch.load(ckpt_path, map_location="cpu")

    cfg_dict = ckpt["config"].copy()
    cfg_dict["vocab_size"] = len(ckpt["stoi"])
    cfg = GPTConfig(**cfg_dict)

    model = GPT(cfg).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    stoi = ckpt["stoi"]
    itos = ckpt["itos"]
    block_size = cfg.block_size
    return model, stoi, itos, block_size

# calculates the zero shot loss, using only 1% of corpus B’s raw text, test model A with its own vocabulary
@torch.no_grad()
def zero_shot_loss(model, stoiA, itosA, block_size, corpusB, frac=0.01, batch_size=32):
    txt = (Path("data") / f"{corpusB}.txt").read_text(encoding="utf-8", errors="ignore")

    dataset = CharDataset(txt, block_size=block_size, stoi=stoiA, itos=itosA)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    take_batches = max(1, math.floor(len(loader) * float(frac)))

    total_loss, seen = 0.0, 0
    if take_batches == 0:
        return None

    model.eval()
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= take_batches:
                break
            x = x.to(device)
            y = y.to(device)
            _, loss = model(x, y)
            total_loss += float(loss.item())
            seen += 1

    return (total_loss / seen) if seen else None

losses = {A: dict.fromkeys(corpora, None) for A in corpora}
generated_samples = []

# create 3x3 loss table and save a sample per pair
for A in corpora:
    mdl, s2i, i2s, blk_sz = load_model(A)
    for B in corpora:
        L = zero_shot_loss(mdl, s2i, i2s, blk_sz, B, frac=0.01, batch_size=32)
        losses[A][B] = L

        prompt_B = prompts[B] 
        txt = generate_small_text(A, prompt_B, "final", 120) or ""
        snip = txt[:120].replace("\n", " ")

        print("{:>12} -> {:<12} | loss ~ {:.3f} | prompt: {!r}".format(A, B, L, prompt_B))
        print("sample:", snip)
        generated_samples.append([A, B, L, prompt_B, snip])

# save 3x3 losses to txt
lines = []
header = ["model_A \\ data_B"] + list(corpora)
lines.append("\t".join(header) + "\n")

for A in corpora:
    fmt = (lambda v: "" if v is None else f"{v:.4f}")
    row_vals = [A] + [fmt(losses[A][B]) for B in corpora]
    lines.append("\t".join(row_vals) + "\n")

with open("zero_shot_losses.txt", "w", encoding="utf-8") as f:
    f.writelines(lines)

print("saved: zero_shot_losses.txt")

# save pairwise generated_samples to csv
lines = []
lines.append("A_model\tB_data\tloss\tprompt\tfirst_120_chars\n")

for a_model, b_data, lval, prmpt, snippet in generated_samples:
    loss_str = "" if lval is None else f"{lval:.4f}"
    row = [a_model, b_data, loss_str, prmpt, snippet]
    lines.append("\t".join(row) + "\n")

with open("zero_shot_generated_samples.txt", "w", encoding="utf-8") as f:
    f.writelines(lines)

print("saved: zero_shot_generated_samples.txt")

## Zero Shot Losses

| model_A \ data_B | shakespeare | wikipedia | math   |
|------------------|-------------|-----------|--------|
| shakespeare      | 1.4454  | 2.3644    | 3.8366 |
| wikipedia        | 2.3098      | 1.4214| 2.7333 |
| math             | 3.2767      | 2.6762    | 1.2729 |


## Zero Shot Generated Samples

| A_model    | shakespeare | shakespeare | shakespeare | wikipedia  | wikipedia  | wikipedia  | math        | math        | math        |
|------------|-------------|-------------|-------------|------------|------------|------------|-------------|-------------|-------------|
| loss       | 1.4454      | 2.3644      | 3.8366      | 2.3098     | 1.4214     | 2.7333     | 3.2767      | 2.6762      | 1.2729      |
| prompt     | What's done can't be undone | This article provides that | Let's take an example of variable x | What's done can't be undone | This article provides that | Let's take an example of variable x | What's done can't be undone | This article provides that | Let's take an example of variable x |
| first_100_chars | What's done can't be undone not honouring Cuped histless: I'll languardent.  DUKE OFrth Warwick's, Where's it wonder, Be | This article provides that what again revers.  Socinorate, welcome,' calls they was at too much would try wosancted He c | Let's take an example of variable xuse.  CORIOLANUS: Haste be force whermine ever ships thus that nohbour!  CAMILLO: Wha | What's done can't be undonessaos between the Brece from SaveD. Hields and London, andores, because adjing defineder two | This article provides that is an iron, influence on the prentined by this imported. People see" unith the sot areas inhe | Let's take an example of variable xister.  The secording repirations, the dishiclusse of the time about 30.07 included J | What's done can't be undone, the " process," and the relation and  the arc AB of area is only 1. 2s/x* = 43 (x 2 +x) 2 = | This article provides that when the arpres  difference will write dx   dy    = x -2      or will an infinite cooss0 2 = | Let's take an example of variable x, so that the effect  value.   (14) ^103a? 3 - i n 2*5 - 26x2+2>x. The difference tra |
| B_data     | shakespeare | wikipedia   | math        | shakespeare | wikipedia  | math        | shakespeare | wikipedia   | math        |


### Few-shot Evaluation

1. Start from a Pretrained Model (Dataset A).

* Choose a model that has already been trained on dataset A (e.g., `wikipedia`, `math`, `shakespeare`).
* Load the final checkpoint for A:

```python
ckpt_path = f"checkpoints/{A}/final.pt"
ckpt = torch.load(ckpt_path, map_location="cpu")
```
* Important: the vocabulary size used for training A may differ from the default.

* Before building the model, set

```python
config.vocab_size = len(ckpt["stoi"])
model = GPT(config)
```
* This ensures the embedding and output layer dimensions match the checkpoint.
* Finally, load the weights:

```python
model.load_state_dict(ckpt["model_state"])
```

2. Fine-tune on Dataset B.

* Use the same vocabulary (`stoi`, `itos`) from dataset A to enco0de dataset B.
* Train for 200 iterations with AdamW (`lr=10, 25, 50, 100, 2ns thereafter if you extend training.
* Naming convention:

```
{A}_start{N}_{B}_run{M}.pt
```

where

* A = source dataset (e.g. `wikipedia`)
* B = target dataset (e.g. `math`)
* N = starting iteration (e.g. `0` if from scratch, or `500` if resuming)
* M = current fine-tuning iteration

3. Evaluate and Track Loss.

* Every 50 steps, evaluate on a 1% subsample of dataset B for speed.
* Record the average evaluation loss.
* Keep a running list of these evaluation losses to plot later.

4. Plot Loss Curves.

* After training, plot loss vs iteration for each experiment.
* Compare curves across different (A→B) fine-tuning runs.
* This shows how quickly and effectively each pretrained model adapts to dataset B.

5. Examples

* At each saved checkpoint (10, 25, 50, 100, 200), generate short text continuations from the model.
* Use the same sampling routine (temperature-controlled decoding).
* Collect and compare generations to see how quality improves over epochs


In [ ]:
import torch
from pathlib import Path
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"

corpora = ["shakespeare", "wikipedia", "math"]
prompts = {
    "shakespeare": "What's done can't be undone",
    "wikipedia": "This article provides that",
    "math": "Let's take an example of variable x",
}

def load_text(corpus):
    return Path(f"data/{corpus}.txt").read_text(encoding="utf-8", errors="ignore")


def find_final_ckpt(corpus):
    try:
        return Path("checkpoints") / corpus / "final.pt"
    except:
        raise FileNotFoundError(f"final.pt not found for {corpus}")

def load_model_from_A(A: str):
    ckpt = torch.load(find_final_ckpt(A), map_location="cpu")
    cfgd = ckpt["config"].copy()
    cfgd["vocab_size"] = len(ckpt["stoi"])
    cfg = GPTConfig(**cfgd)
    model = GPT(cfg).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.train()
    return model, ckpt["stoi"], ckpt["itos"], cfg.block_size, cfg

@torch.no_grad()
def eval_on_subset(model, ds, frac: float = 0.01, batch_size: int = 32):
    model.eval()
    loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
    max_batches = max(1, int(len(loader) * frac))
    losses = []
    for i, (xb, yb) in enumerate(loader):
        if i >= max_batches: break
        xb, yb = xb.to(device), yb.to(device)
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return float(sum(losses) / len(losses))

@torch.no_grad()
def sample_with_model(model, stoi, itos, prompt, block_size, max_new_tokens=120, temperature=0.9, top_k=50):
    x = torch.tensor([[stoi.get(c, 0) for c in prompt]], dtype=torch.long, device=device)
    idx = model.generate(x, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return "".join(itos[i] for i in idx[0].tolist())

def run_fewshot(A, B, steps = 200, eval_every = 50, save_steps = (10, 25, 50, 100, 200), lr = 1e-3):
    modelA, stoiA, itosA, block_sizeA, cfgA = load_model_from_A(A)

    dsB = CharDataset(load_text(B), block_size=block_sizeA, stoi=stoiA, itos=itosA)
    loaderB = DataLoader(dsB, batch_size=32, shuffle=True)
    optimizer = torch.optim.AdamW(modelA.parameters(), lr=lr)

    out_dir = Path("checkpoints_fewshot"); out_dir.mkdir(parents=True, exist_ok=True)
    loss_log, samples = [], []

    it_loader = iter(loaderB)
    for step in range(1, steps + 1):
        try: xb, yb = next(it_loader)
        except StopIteration:
            it_loader = iter(loaderB); xb, yb = next(it_loader)
        xb, yb = xb.to(device), yb.to(device)

        _, loss = modelA(xb, yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

        eval_loss = None
        if step % eval_every == 0:
            eval_loss = eval_on_subset(modelA, dsB, frac=0.01, batch_size=32)

        loss_log.append((step, eval_loss, float(loss.item())))

        if step in save_steps:
            ckpt_name = f"{A}_start0_{B}_run{step}.pt"
            torch.save(
                {"model_state": modelA.state_dict(), "config": cfgA.__dict__, "stoi": stoiA,
                 "itos": itosA, "A": A, "B": B, "startN": 0, "runM": step},
                out_dir / ckpt_name,
            )
            s = sample_with_model(modelA, stoiA, itosA, prompts[B], block_sizeA, max_new_tokens=120)
            samples.append((step, prompts[B], s[:120].replace("\n", " ")))

    logs_dir = Path("logs")
    logs_dir.mkdir(parents=True, exist_ok=True)

    loss_txt = logs_dir / f"fewshot_{A}_to_{B}_losses.txt"
    with open(loss_txt, "w", encoding="utf-8") as f:
        f.write("step\teval_1pct_loss_or_blank\ttrain_loss\n")
        for st, ev, tr in loss_log:
            ev_str = "" if ev is None else f"{ev:.4f}"
            f.write(f"{st}\t{ev_str}\t{tr:.4f}\n")

    samp_txt = logs_dir / f"fewshot_{A}_to_{B}_samples.txt"
    with open(samp_txt, "w", encoding="utf-8") as f:
        f.write("step\tprompt\tfirst_120_chars\n")
        for st, pr, sv in samples:
            f.write(f"{st}\t{pr}\t{sv}\n")

    plots_dir = Path("plots")
    plots_dir.mkdir(parents=True, exist_ok=True)

    steps_x = [st for st, _, _ in loss_log]
    train_y = [tr for _, _, tr in loss_log]
    eval_pts = [(st, ev) for st, ev, _ in loss_log if ev is not None]

    plt.figure(figsize=(6.5, 4))
    plt.plot(steps_x, train_y, label="train loss", marker="o", linewidth=1)
    if eval_pts:
        es, evs = zip(*eval_pts)
        plt.plot(es, evs, label="eval loss (1%)", marker="s", linewidth=1)
    plt.title(f"Few-shot Loss: {A} \u2192 {B}")
    plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.tight_layout()

    out_png = plots_dir / f"fewshot_{A}_to_{B}_loss_plot.png"
    plt.savefig(str(out_png), dpi=200, bbox_inches="tight")
    plt.show()

for A in corpora:
    for B in corpora:
        if B == A: continue
        run_fewshot(A, B, steps=200, eval_every=50, save_steps=(10, 25, 50, 100, 200), lr=1e-3)

## Few-shot Evaluation Results
### General Observations
- All models show an initial sharp drop in loss (first ~20–30 steps), then slower convergence.
- Eval loss (orange) vs train loss (blue):
- When they track closely → good adaptation.
- When eval stays higher → difficult adaptation.
- Final eval losses stabilize between 1.6–2.0, but some domains need lower loss for convincing text.

### Run-by-Run Analysis
#### Math → Shakespeare
- Loss: 3.3 → 1.7.
- Train and eval closely aligned → strong transfer.
- Math’s structure helps adapt to Shakespeare’s repetitive style.
- Quality threshold: ~1.8.

#### Math → Wikipedia
- Loss: 2.8 → 1.6–1.7.
- Eval matches train → strong adaptation.
- Wikipedia is broader but still benefits from math pretraining.
- Quality threshold: ~1.7.

#### Shakespeare → Math
- Train: 4.0 → 1.5, but eval ~2.0 (gap).
- Indicates overfitting to Shakespeare style.
- Math requires <1.4 for convincing notation → not reached.
- Weak transfer.

#### Shakespeare → Wikipedia
- Loss: 2.2 → 1.6.
- Eval follows train well → moderate-to-good transfer.
- Narrow Shakespeare vocabulary doesn’t handicap too much.
- Quality threshold: ~1.6–1.7.

#### Wikipedia → Math
- Loss: 2.5 → 1.6, but eval ~2.0 (gap).
- Same issue as Shakespeare → Math: math is unforgiving.
- Moderate but not sufficient adaptation.

#### Wikipedia → Shakespeare
- Loss: 2.4 → 1.6.
- Train and eval closely aligned → excellent transfer.
- Wikipedia’s broad coverage adapts easily to Shakespeare’s style.
- Best-performing transfer overall.

### Cross-Domain Conclusions
- Easiest target domain:
- Shakespeare (formulaic, forgiving, convincing at ~1.6–1.7 loss).
- Hardest target domain:
- Math (rigid, unforgiving, requires <1.4 loss).
- Most versatile source domain:
- Wikipedia → adapts well to Shakespeare, moderately to Math.
- Least versatile source domain:
- Shakespeare → poor adaptation to Math, only moderate to Wikipedia.


### Few-Shot Transfer From Earlier Checkpoints

I repeat the adaptation experiments from partially trained source models to test whether earlier checkpoints transfer more easily than fully trained ones. This section is mainly about understanding whether incomplete source specialization helps or hurts downstream adaptation.


In [ ]:
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"

start_iters, steps, LR, evaluation_step, required_steps = [500, 1000], 200, 1e-3, 50, (10, 25, 50, 100, 200)
corpora = ["shakespeare", "wikipedia", "math"]
prompts = {
    "shakespeare": "What's done can't be undone",
    "wikipedia": "This article provides that",
    "math": "Let's take an example of variable x",
}

def load_text(corpus):
    return Path(f"data/{corpus}.txt").read_text(encoding="utf-8", errors="ignore")

def find_iter_ckpt(corpus, it):
    try:
        if iter == "final":
            return Path("checkpoints") / corpus / "final.pt"
        else:
            return Path("checkpoints") / corpus / f"iter_{it}.pt"
    except:
        raise FileNotFoundError(f"final.pt not found for {corpus}")

def load_model_from_iter(A, start_iter):
    ckpt = torch.load(find_iter_ckpt(A, start_iter), map_location="cpu")
    cfgd = ckpt["config"].copy(); cfgd["vocab_size"] = len(ckpt["stoi"])
    cfg = GPTConfig(**cfgd)
    model = GPT(cfg).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.train()
    return model, ckpt["stoi"], ckpt["itos"], cfg.block_size, cfg

@torch.no_grad()
def eval_on_subset(model, dataset, frac=0.01, batch_size=32):
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    max_batches = max(1, int(len(loader) * frac))
    losses = []
    for i, (xb, yb) in enumerate(loader):
        if i >= max_batches: break
        xb, yb = xb.to(device), yb.to(device)
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return float(sum(losses)/len(losses)) if losses else None

@torch.no_grad()
def sample_text(model, stoi, itos, prompt, block_size, max_new_tokens=120, temperature=0.9, top_k=50):
    x = torch.tensor([[stoi.get(c, 0) for c in prompt]], dtype=torch.long, device=device)
    idx = model.generate(x, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return "".join(itos[i] for i in idx[0].tolist())

class CharDataset(torch.utils.data.Dataset):
    def __init__(self, text, block_size, stoi, itos):
        self.stoi, self.itos = stoi, itos
        self.block_size = block_size
        self.data = torch.tensor([self.stoi.get(c, 0) for c in text], dtype=torch.long)
    def __len__(self): return len(self.data) - self.block_size
    def __getitem__(self, idx):
        chunk = self.data[idx: idx+self.block_size+1]
        return chunk[:-1], chunk[1:]

def run_fewshot_from_early(A, B, start_iter):
    model, stoi, itos, block_size, cfg = load_model_from_iter(A, start_iter)

    dsB = CharDataset(load_text(B), block_size=block_size, stoi=stoi, itos=itos)
    loaderB = DataLoader(dsB, batch_size=32, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    it_loader = iter(loaderB)

    ckpt_dir = Path("checkpoints_fewshot_step3"); ckpt_dir.mkdir(parents=True, exist_ok=True)
    logs_dir = Path("logs"); logs_dir.mkdir(parents=True, exist_ok=True)
    plots_dir = Path("plots"); plots_dir.mkdir(parents=True, exist_ok=True)

    losses_log, samples_log = [], []

    for step in range(1, steps+1):
        try:
            xb, yb = next(it_loader)
        except StopIteration:
            it_loader = iter(loaderB)
            xb, yb = next(it_loader)

        xb, yb = xb.to(device), yb.to(device)
        _, loss = model(xb, yb)
        opt.zero_grad(); loss.backward(); opt.step()

        eval_loss = None
        if step % evaluation_step == 0:
            eval_loss = eval_on_subset(model, dsB, frac=0.01, batch_size=32)

        losses_log.append((step, eval_loss, float(loss.item())))

        if step in required_steps:
            name = f"{A}_start{start_iter}_{B}_run{step}.pt"
            torch.save(
                {"model_state": model.state_dict(), "config": cfg.__dict__,
                 "stoi": stoi, "itos": itos, "A": A, "B": B, "startN": start_iter, "runM": step},
                ckpt_dir / name,
            )
            s = sample_text(model, stoi, itos, prompts[B], block_size, max_new_tokens=120)
            samples_log.append((step, prompts[B], s[:120].replace("\n", " ")))

    loss_txt = logs_dir / f"step3_{A}_iter{start_iter}_to_{B}_losses.txt"
    with open(loss_txt, "w", encoding="utf-8") as f:
        f.write("step\teval_1pct_loss_or_blank\ttrain_loss\n")
        for st, ev, tr in losses_log:
            f.write(f"{st}\t{'' if ev is None else f'{ev:.4f}'}\t{tr:.4f}\n")

    samp_txt = logs_dir / f"step3_{A}_iter{start_iter}_to_{B}_samples.txt"
    with open(samp_txt, "w", encoding="utf-8") as f:
        f.write("step\tprompt\tfirst_120_chars\n")
        for st, pr, sv in samples_log:
            f.write(f"{st}\t{pr}\t{sv}\n")

    xs = [st for st,_,_ in losses_log]
    tr = [tl for _,_,tl in losses_log]
    ev = [(st,evl) for st,evl,_ in losses_log if evl is not None]
    plt.figure(figsize=(6.5,4))
    plt.plot(xs, tr, label="train loss", marker="o", linewidth=1)
    if ev:
        es, evs = zip(*ev)
        plt.plot(es, evs, label="eval loss (1%)", marker="s", linewidth=1)
    plt.title(f"Few-shot: {A}@{start_iter} → {B}")
    plt.xlabel("step"); plt.ylabel("loss"); plt.legend()
    plt.tight_layout()
    out_png = plots_dir / f"step3_{A}_iter{start_iter}_to_{B}_loss_plot.png"
    plt.savefig(out_png, dpi=200, bbox_inches="tight")
    plt.show()

for A in corpora:
    for B in corpora:
        if A == B: continue
        for start_iter in start_iters:
            try:
                run_fewshot_from_early(A, B, start_iter)
            except Exception as e:
                print(f"Skip {A}@{start_iter} -> {B}: {e}")


### Partial training does not make adaptation easier. In fact, adaptation is harder when starting from earlier checkpoints, since the model hasn’t learned enough structure yet. The longer the source model is pretrained (1000 > 500 > full training), the better it adapts to a new target. This highlights the benefit of strong pretraining before fine-tuning.


## Zero Shot Losses

| model_A \ data_B | shakespeare | wikipedia | math   |
|------------------|-------------|-----------|--------|
| shakespeare      | 1.4454  | 2.3644    | 3.8366 |
| wikipedia        | 2.3098      | 1.4214| 2.7333 |
| math             | 3.2767      | 2.6762    | 1.2729 |

## Plots
Every code block output has the plots. Otherwise, please check the plots folder

## Samples

## Zero Shot Generated Samples

| A_model    | shakespeare | shakespeare | shakespeare | wikipedia  | wikipedia  | wikipedia  | math        | math        | math        |
|------------|-------------|-------------|-------------|------------|------------|------------|-------------|-------------|-------------|
| loss       | 1.4454      | 2.3644      | 3.8366      | 2.3098     | 1.4214     | 2.7333     | 3.2767      | 2.6762      | 1.2729      |
| prompt     | What's done can't be undone | This article provides that | Let's take an example of variable x | What's done can't be undone | This article provides that | Let's take an example of variable x | What's done can't be undone | This article provides that | Let's take an example of variable x |
| first_100_chars | What's done can't be undone not honouring Cuped histless: I'll languardent.  DUKE OFrth Warwick's, Where's it wonder, Be | This article provides that what again revers.  Socinorate, welcome,' calls they was at too much would try wosancted He c | Let's take an example of variable xuse.  CORIOLANUS: Haste be force whermine ever ships thus that nohbour!  CAMILLO: Wha | What's done can't be undonessaos between the Brece from SaveD. Hields and London, andores, because adjing defineder two | This article provides that is an iron, influence on the prentined by this imported. People see" unith the sot areas inhe | Let's take an example of variable xister.  The secording repirations, the dishiclusse of the time about 30.07 included J | What's done can't be undone, the " process," and the relation and  the arc AB of area is only 1. 2s/x* = 43 (x 2 +x) 2 = | This article provides that when the arpres  difference will write dx   dy    = x -2      or will an infinite cooss0 2 = | Let's take an example of variable x, so that the effect  value.   (14) ^103a? 3 - i n 2*5 - 26x2+2>x. The difference tra |
| B_data     | shakespeare | wikipedia   | math        | shakespeare | wikipedia  | math        | shakespeare | wikipedia   | math        |


## Discussion
- When probabilities sharpen: After some training, models begin assigning one token a dominant probability. Early on, distributions are wide, but with more steps they narrow down.
- When probabilities stay broad: On mismatched prompts (e.g., math model with Shakespeare text), the token probabilities remain spread out longer, reflecting greater uncertainty.
- In-domain vs out-of-domain: Models trained on the same domain reach low loss quickly and gain confidence, while out-of-domain cases require lower losses before the output appears correct.
- Corpus-specific thresholds: The Shakespeare model achieved convincing style around 1.6–1.8 loss, whereas Wikipedia and Math needed losses closer to 1.5 or below for outputs to seem accurate.
- Domains that sound good early: Shakespeare’s language started resembling plays quickly, while Wikipedia and Math took more training before they looked natural.
- Token biases: Certain tokens, like spaces and frequent letters (e, t, o, n), consistently had high probabilities. For math, domain-specific symbols such as x, y, d, and + appeared more often.
- Overall insight: Zero-shot transfer is weak across domains, but few-shot fine-tuning improves adaptation. Less-trained checkpoints can shift domains more easily, while fully trained models become more specialized and less flexible.


In [ ]:
import torch
import torch.nn.functional as F
import os
import matplotlib.pyplot as plt
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

def load_model(ckpt_path,  GPT):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    config = type("GPTConfig", (), ckpt["config"])
    stoi, itos = ckpt["stoi"], ckpt["itos"]
    config.vocab_size = len(stoi)
    model = GPT(config)
    model.load_state_dict(ckpt["model_state"])
    return model, stoi, itos

def inspect_softmax(model, prompt, stoi, itos, top_k=10):
    # pick a fallback index if <unk> not in vocab
    unk_idx = stoi.get("<unk>", next(iter(stoi.values())))  

    x = torch.tensor([[stoi.get(c, unk_idx) for c in prompt]], dtype=torch.long)
    with torch.no_grad():
        probs = F.softmax(model(x)[0][:, -1, :], dim=-1).squeeze()
    entropy = -(probs * torch.log(probs + 1e-12)).sum().item()
    top_p, top_i = torch.topk(probs, top_k)
    return {
        "prompt": prompt,
        "entropy": entropy,
        "probs": [(itos[i.item()], p.item()) for i, p in zip(top_i, top_p)]
    }


In [ ]:
import collections
import matplotlib.pyplot as plt
from pathlib import Path

corpora = ["shakespeare", "wikipedia", "math"]
prompts = {
    "shakespeare": "What's done can't be undone",
    "wikipedia": "This article provides that",
    "math": "Let's take an example of variable x",
}
iters, top_k, plot_k = [0, 500, 1000, 1500, 2000, 3000, 4000, 4500, "final"], 10, 5

def find_ckpt(corpus, it):
    p = Path("checkpoints")/corpus/("final.pt" if it == "final" else f"iter_{it}.pt")
    return p if p.exists() else None

def iter_x(it): 
    return 999999 if it == "final" else int(it)

logs_dir  = Path("logs");  logs_dir.mkdir(parents=True, exist_ok=True)
plots_dir = Path("plots"); plots_dir.mkdir(parents=True, exist_ok=True)

for model_name in corpora:
    for prompt_name, prompt_text in prompts.items():
        rows, xs, ents = [], [], []
        for it in iters:
            ck = find_ckpt(model_name, it)
            if not ck: 
                continue
            try:
                model, stoi, itos = load_model(str(ck), GPT)
                out = inspect_softmax(model, prompt_text, stoi, itos, top_k=top_k)
            except Exception as e:
                print(f"skipping {model_name}@{it} due to: {e}")
                continue

            entropy, probs = out["entropy"], out["probs"]
            xs.append(iter_x(it)); ents.append(entropy)

            flat = [x for tok, p in probs[:top_k] for x in (tok, f"{p:.4f}")]
            rows.append([model_name, prompt_name, it, f"{entropy:.4f}", *flat])

        out_txt = logs_dir / f"softmax_{model_name}_{prompt_name}.txt"
        header = ["model", "prompt", "iter", "entropy"] + \
                 [y for k in range(1, top_k+1) for y in (f"top{k}_token", f"top{k}_prob")]
        with open(out_txt, "w", encoding="utf-8") as f:
            f.write("\t".join(header) + "\n")
            for r in rows:
                if len(r) < len(header):
                    r = r + [""] * (len(header) - len(r))
                f.write("\t".join(map(str, r)) + "\n")

        if xs:
            X, Y = zip(*sorted(zip(xs, ents)))
            plt.figure(figsize=(6, 4))
            plt.plot(X, Y, marker="o", linewidth=1)
            plt.title(f"Entropy vs Iter ({model_name}, prompt={prompt_name})")
            plt.xlabel("iteration (final = 999999)")
            plt.ylabel("entropy")
            plt.tight_layout()
            out_png = plots_dir / f"softmax_entropy_{model_name}_{prompt_name}.png"
            plt.savefig(out_png, dpi=200, bbox_inches="tight")
            plt.show()

        per_iter, token_hits = [], collections.Counter()
        for it in iters:
            ck = find_ckpt(model_name, it)
            if not ck:
                continue
            try:
                model, stoi, itos = load_model(str(ck), GPT)
                out = inspect_softmax(model, prompt_text, stoi, itos, top_k=top_k)
            except Exception:
                continue
            tok_probs = dict(out["probs"])
            per_iter.append((it, tok_probs))
            token_hits.update(tok_probs)

        if not per_iter:
            continue

        keep_tokens = [t for t, _ in token_hits.most_common(plot_k)]
        X = [iter_x(it) for it, _ in per_iter]
        series = {t: [float(tp.get(t, 0.0)) for _, tp in per_iter] for t in keep_tokens}

        out_txt2 = logs_dir / f"top_tokens_{model_name}_{prompt_name}.txt"
        with open(out_txt2, "w", encoding="utf-8") as f:
            f.write("\t".join(["iter"] + keep_tokens) + "\n")
            for i, (it, _) in enumerate(per_iter):
                row = [str(it)] + [f"{series[t][i]:.4f}" for t in keep_tokens]
                f.write("\t".join(row) + "\n")

        plt.figure(figsize=(7, 4))
        for t in keep_tokens:
            plt.plot(X, series[t], marker="o", linewidth=1, label=repr(t))
        plt.title(f"Top-token probs vs Iter ({model_name}, prompt={prompt_name})")
        plt.xlabel("iteration (final = 999999)")
        plt.ylabel("probability")
        if keep_tokens:
            plt.legend(fontsize=8)
        plt.tight_layout()
        out_png2 = plots_dir / f"top_tokens_{model_name}_{prompt_name}.png"
        plt.savefig(out_png2, dpi=200, bbox_inches="tight")
        plt.show()


- When looking at the graphs, a clear pattern emerges. In the in-domain cases — such as the Shakespeare model reading Shakespeare, the Wikipedia model on Wikipedia, or the Math model on Math — the probabilities harden quickly. The most likely token becomes dominant almost right away, and the entropy curves fall sharply. This shows that the models are confident in their predictions, largely because the sequences follow the patterns they were trained on: poetic lines in Shakespeare, factual exposition in Wikipedia, and formulaic symbols in mathematics. In these familiar contexts, the models settle into certainty very early in the generation process.

- The picture changes in the out-of-domain cases. When a Shakespeare model tries to continue a math passage, or when a Math model is faced with Wikipedia prose, the distributions remain far more diffuse. The graphs show probabilities spread across several competing tokens, and the entropy curves stay high and unstable. The models hesitate, unable to lock onto a single prediction. Even when a dominant token eventually emerges, it does so much more slowly, and in some cases it never fully stabilizes. This reflects the models’ difficulty in handling text that lies outside their learned statistical structures.

- These findings match what we would expect. Each model has internalized the style and structure of its training data, making it comfortable and confident in that space. Shakespeare’s model handles literary rhythm, Wikipedia’s model is tuned to encyclopedic explanation, and the Math model anticipates symbols and equations. But when the input comes from a different domain, the statistical cues don’t line up, and the models are left uncertain.

- The implications are twofold. On the one hand, predictability is strong in-domain: the models “know” what is coming next and produce confident, fluent continuations. On the other hand, predictability breaks down in out-of-domain settings, where continuations feel awkward and probabilities remain diffuse. This highlights a key trade-off. The models are highly specialized within their domains, but that specialization comes at the cost of flexibility. Without additional fine-tuning or examples, they cannot easily generalize across different styles of text.


## Gradient-Based Token Attribution

I use gradient norms on token embeddings as a lightweight interpretability signal for next-token prediction. Instead of treating this as a generic exercise, I use it to compare how token importance changes across prompts, checkpoints, and domain-specific models.

The main questions in this section are:

- which input tokens most influence the next-token distribution
- how attribution changes as training progresses
- whether different domains rely on different token patterns or structural cues
- where gradient-based explanations remain useful and where they become misleading


In [ ]:
import torch
import torch.nn.functional as F

import os
import matplotlib.pyplot as plt
import numpy as np

def token_gradients(model, prompt, target_token, stoi, itos ):
    """
    Compute gradient norms per input token wrt probability of `target_token`.
    """
    model.eval()
    x = torch.tensor([stoi[c] for c in prompt], dtype=torch.long)[None, :] 

    grads = {}

    # Hook to capture embeddings with gradient
    def save_grad(module, inp, out):
        out.retain_grad()
        grads['emb'] = out

    handle = model.transformer.wte.register_forward_hook(save_grad)

    # Forward pass
    logits, _ = model(x)
    last_logits = logits[:, -1, :]   # [B, vocab]

    target_idx = stoi.get(target_token, None)
    if target_idx is None:
        raise ValueError(f"Token {target_token!r} not in vocab")

    probs = F.softmax(last_logits, dim=-1)
    target_prob = probs[0, target_idx]

    # Backprop
    model.zero_grad(set_to_none=True)
    target_prob.backward()

    # Now grads['emb'] has the embedding grads
    grad_norms = grads['emb'].grad[0].norm(dim=-1)  # [T]
    tokens = [itos[i.item()] for i in x[0]]

    # Clean up the hook
    handle.remove()

    return list(zip(tokens, grad_norms.tolist()))



In [ ]:
import os, glob, torch, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

corpora = ["shakespeare", "wikipedia", "math"]
target_tokens = ["e", " ", ":", "x", "y", "."]
zero_prompts = {
    "shakespeare": "What's done can't be undone",
    "wikipedia": "This article provides that",
    "math": "Let's take an example of variable x",
}
few_prompts = {
    "shakespeare": (
        "HAM: Wherefore art thou?\n"
        "OTH: Speak plainly.\n"
        "Now: To be, or not to be: "
    ),
    "wikipedia": (
        "Ex1: The Moon orbits Earth.\n"
        "Ex2: Photosynthesis converts light to chemical energy.\n"
        "Now: In this article, "
    ),
    "math": (
        "Ex1: Let g(x)=2x+1.\n"
        "Ex2: Suppose g(0)=1.\n"
        "Now: Let f be a function "
    ),
}

logs_dir  = Path("logs");  logs_dir.mkdir(parents=True, exist_ok=True)
plots_dir = Path("plots"); plots_dir.mkdir(parents=True, exist_ok=True)

def load_model_from_ckpt(ckpt_path, GPT):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    cfg = ckpt["config"].copy()
    cfg["vocab_size"] = len(ckpt["stoi"])
    Cfg = type("GPTConfig", (), cfg)
    model = GPT(Cfg).to(DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model, ckpt["stoi"], ckpt["itos"], Cfg.block_size

def list_ckpts_every_500(corpus):
    base = Path("checkpoints") / corpus
    if not base.exists():
        return []
    paths = []
    for p in sorted(glob.glob(str(base / "iter_*.pt"))):
        try:
            step = int(os.path.basename(p).split("_")[1].split(".")[0])
            if step % 500 == 0:
                paths.append(p)
        except:
            pass
    final_pt = base / "final.pt"
    if final_pt.exists():
        paths.append(str(final_pt))
    return paths

def get_step_from_name(name):
    if name.startswith("iter_") and name.endswith(".pt"):
        return int(name.split("_")[1].split(".")[0])
    return 5000  # final

def make_safe_prompt(prompt, stoi):
    fallback = next(iter(stoi.keys()))
    return "".join([c if c in stoi else fallback for c in prompt])

@torch.no_grad()
def last_token_entropy(model, prompt, stoi):
    unk = next(iter(stoi.values()))
    x = torch.tensor([[stoi.get(c, unk) for c in prompt]], dtype=torch.long).to(DEVICE)
    logits, _ = model(x)
    probs = torch.softmax(logits[:, -1, :], dim=-1).squeeze(0)
    ent = -(probs * torch.log(probs + 1e-12)).sum().item()
    return ent

for corpus in corpora:
    ckpts = list_ckpts_every_500(corpus)
    if not ckpts:
        print(f"[skip] no checkpoints found for {corpus}")
        continue

    zs_prompt = zero_prompts[corpus]
    fs_prompt = few_prompts[corpus]

    grads_by_tgt = {t: [] for t in target_tokens}
    x_tokens = None

    for ckpt_path in ckpts:
        step = get_step_from_name(os.path.basename(ckpt_path))
        model, stoi, itos, block_size = load_model_from_ckpt(ckpt_path, GPT)

        safe_zs_full = make_safe_prompt(zs_prompt, stoi)
        safe_zs = safe_zs_full[: max(1, block_size - 1)]

        for tgt in target_tokens:
            if tgt not in stoi:
                continue
            pairs = token_gradients(model, safe_zs, tgt, stoi, itos)
            tokens_now = [p[0] for p in pairs]
            grads_now  = [p[1] for p in pairs]

            if x_tokens is None:
                x_tokens = tokens_now
            else:
                if len(tokens_now) != len(x_tokens):
                    m = min(len(tokens_now), len(x_tokens))
                    tokens_now = tokens_now[:m]; grads_now = grads_now[:m]; x_tokens = x_tokens[:m]

            grads_by_tgt[tgt].append((step, grads_now))

    for tgt in target_tokens:
        rows = grads_by_tgt[tgt]
        if not rows:
            continue
        rows.sort(key=lambda x: x[0])
        steps, grads_lists = zip(*rows)
        M = np.array(grads_lists)
        M_norm = M / (M.max(axis=1, keepdims=True) + 1e-9)

        plt.figure(figsize=(max(6, 0.25*len(x_tokens)), 0.5*len(steps) + 2))
        plt.imshow(M_norm, aspect="auto", interpolation="nearest")
        cbar = plt.colorbar()
        cbar.set_label("normalized grad-norm")
        plt.yticks(range(len(steps)), steps)
        plt.xticks(range(len(x_tokens)), x_tokens, rotation=90)
        plt.title(f"{corpus} | ribbon | tgt='{tgt}'")
        plt.xlabel("input tokens (zero-shot prompt)")
        plt.ylabel("checkpoint step (sorted)")
        plt.tight_layout()
        out_png_ribbon = plots_dir / f"{corpus}_ribbon_tgt_{ord(tgt)}.png"
        plt.savefig(out_png_ribbon, dpi=200, bbox_inches="tight")
        plt.show()

    last_ckpt = ckpts[-1]
    model, stoi, itos, block_size = load_model_from_ckpt(last_ckpt, GPT)
    safe_zs = make_safe_prompt(zs_prompt, stoi)[: max(1, block_size - 1)]
    safe_fs = make_safe_prompt(fs_prompt, stoi)[: max(1, block_size - 1)]

    for tgt in target_tokens:
        if tgt not in stoi:
            continue
        zs_pairs = token_gradients(model, safe_zs, tgt, stoi, itos)
        fs_pairs = token_gradients(model, safe_fs, tgt, stoi, itos)
        zs_tokens, zs_grads = [p[0] for p in zs_pairs], [p[1] for p in zs_pairs]
        fs_tokens, fs_grads = [p[0] for p in fs_pairs], [p[1] for p in fs_pairs]

        zs_txt = logs_dir / f"zs_{corpus}_tgt{ord(tgt)}.txt"
        with open(zs_txt, "w", encoding="utf-8") as f:
            f.write("token\tgrad_norm\n")
            for t, g in zip(zs_tokens, zs_grads):
                f.write(f"{t}\t{g}\n")

        fs_txt = logs_dir / f"fs_{corpus}_tgt{ord(tgt)}.txt"
        with open(fs_txt, "w", encoding="utf-8") as f:
            f.write("token\tgrad_norm\n")
            for t, g in zip(fs_tokens, fs_grads):
                f.write(f"{t}\t{g}\n")

        plt.figure(figsize=(max(6, 0.25*len(zs_tokens)), 3))
        plt.bar(range(len(zs_tokens)), zs_grads)
        plt.xticks(range(len(zs_tokens)), zs_tokens, rotation=90)
        plt.title(f"{corpus} ZS @ final | tgt='{tgt}'")
        plt.tight_layout()
        zs_png = plots_dir / f"{corpus}_final_ZERO_tgt_{ord(tgt)}.png"
        plt.savefig(zs_png, dpi=200, bbox_inches="tight")
        plt.show()

        plt.figure(figsize=(max(6, 0.25*len(fs_tokens)), 3))
        plt.bar(range(len(fs_tokens)), fs_grads)
        plt.xticks(range(len(fs_tokens)), fs_tokens, rotation=90)
        plt.title(f"{corpus} FS @ final | tgt='{tgt}'")
        plt.tight_layout()
        fs_png = plots_dir / f"{corpus}_final_FEW_tgt_{ord(tgt)}.png"
        plt.savefig(fs_png, dpi=200, bbox_inches="tight")
        plt.show()

    zs_ent, fs_ent, steps = [], [], []
    for ckpt_path in ckpts:
        step = get_step_from_name(os.path.basename(ckpt_path))
        model, stoi, itos, block_size = load_model_from_ckpt(ckpt_path, GPT)
        zs_ent.append(last_token_entropy(model, make_safe_prompt(zs_prompt, stoi)[:max(1,block_size-1)], stoi))
        fs_ent.append(last_token_entropy(model, make_safe_prompt(fs_prompt, stoi)[:max(1,block_size-1)], stoi))
        steps.append(step)

    order = np.argsort(steps)
    steps = list(np.array(steps)[order])
    zs_ent = list(np.array(zs_ent)[order])
    fs_ent = list(np.array(fs_ent)[order])

    plt.figure(figsize=(6.5, 3.2))
    plt.plot(steps, zs_ent, marker="o", label="zero-shot")
    plt.plot(steps, fs_ent, marker="o", label="few-shot")
    plt.title(f"{corpus} | last-token entropy over checkpoints")
    plt.xlabel("checkpoint step")
    plt.ylabel("entropy")
    plt.legend()
    plt.tight_layout()
    ent_png = plots_dir / f"{corpus}_entropy_over_ckpts.png"
    plt.savefig(ent_png, dpi=200, bbox_inches="tight")
    plt.show()

### 1. Character influence
- #### Who gets the biggest gradient norms?
Consistently the most recent characters right before the target dominate.
- Examples:
- Wikipedia zero-shot (target “ ” and also “x/y”): the bars for the final word “that” (t-h-a-t) are largest.
- Math zero-shot (target “x” or “ ”): the last few characters in “…variable x” spike, with “e”, “x” towering over earlier tokens.
#### Vowels vs consonants vs punctuation/digits:
It’s position > type. Both vowels and consonants win when they are near the target. Punctuation only “wins” when it is the target or adjacent (e.g., the colon “:” ribbon in math lights up the trailing characters). Digits/symbols matter in math prompts when they’re close (e.g., “=”, “:”, variable letters).
#### Frequent vs rare characters:
Rare, task-diagnostic characters (e.g., “x” in math) can outrank frequent letters. Frequent letters like “e” do show moderate signal, but they don’t dominate unless they’re in the last word.

### 2. Checkpoint dynamics
- The last-token entropy curves drop fast early in training, then flatten.
- Wikipedia: zero-shot entropy falls below ~0.3 by ~1k steps and stays very low; few-shot remains higher than ZS but still declines early.
- Math: zero-shot drops to ~0.8–1.0 mid-training, then rises a bit and finishes ~1.3; few-shot goes ~0 by ~1.5k and hugs ~0 afterward.
- Shakespeare: both decline quickly; few-shot is consistently lower than zero-shot.
- The ribbon heatmaps show influence getting more concentrated on the final few characters as training progresses (the bright band shifts to the right edge).

### 3. Domain differences
- Wikipedia model: relies heavily on function/connector characters in the last word (“that”) and surrounding spaces—clean, prose-like locality.
- Math model: emphasizes structure and symbols—peaks on variable letters (“x”), delimiters (space, “:”), and nearby operators; those outshine earlier alphabetic context.
- Shakespeare model: again shows last-word focus, but influence is spread across word-final vowels/consonants that complete Elizabethan-style words; still locality-driven, less symbol-driven than math.

### 4. Zero-shot vs few-shot
- In the bar plots at the final checkpoint, the few-shot gradients are much smaller in magnitude (note the y-axis scales) and tighter around the final tokens.
- Entropy is also lower under few-shot across domains (especially math), indicating more confident / less diffuse predictions.
- The few-shot exemplars seem to anchor the model, reducing uncertainty so that fewer earlier characters need to move the loss.

### 5. Interpretability limits of gradient norms
- Local and myopic: Character-level gradients reflect immediate next-token error, not global sequence importance. A token can be crucial for meaning yet show a small gradient if the next character is already predictable.
- Uncertainty vs importance: Large gradients often indicate model uncertainty, not semantic “importance.” In your plots, early training or zero-shot settings raise gradients even on bland tokens.
- Tokenization artifacts: With character tokens, morphology and spaces dominate—this can overstate the importance of whitespace and suffix letters.
- Saturation & scale: If the model is already confident (few-shot, late training), gradients shrink—but those inputs might still be vital; the gradient just isn’t informative anymore.
- Interactions are hidden: Norms collapse layer-wise and head-wise dynamics; they miss nonlinear feature interactions that actually drive predictions.


## Results

The experiments showed clear domain specialization across Shakespeare, Wikipedia, and math corpora, with noticeable differences in convergence behavior, sample quality, and cross-domain transfer. Few-shot adaptation improved performance relative to zero-shot evaluation, but the gains depended strongly on the source checkpoint and the structural similarity between domains.

## Takeaways

This project strengthened my understanding of small-transformer training, transfer behavior, and lightweight interpretability. The main lesson was that evaluation has to connect loss, generation quality, and token-level behavior rather than relying on a single metric.
